# OpenDub — Gradio dubbing backend (Google Colab)

Run the cells from top to bottom. In Runtime → Change runtime type, select a T4 GPU when available. The final cell launches the Gradio backend with `share=True`; copy its public URL into the separate browser studio. The browser uploads only extracted WAV audio, not your original video. Start with a 10–30 second clip. The share URL is public to anyone who has it, and Colab sessions and free GPU availability are temporary—do not process private media.

In [ ]:
from pathlib import Path

REPO = Path('/content/PlayGround')
if not (REPO / 'app.py').exists():
    !git clone https://github.com/moonedition7-alt/PlayGround.git {REPO}
%cd /content/PlayGround

In [ ]:
# Install system media tools and the backend dependencies.
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

In [ ]:
# Use faster-whisper small with float16 on GPU; use int8 on CPU fallback.
import os
import torch

has_gpu = torch.cuda.is_available()
os.environ['WHISPER_DEVICE'] = 'cuda' if has_gpu else 'cpu'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16' if has_gpu else 'int8'
os.environ['HOST'] = '0.0.0.0'
os.environ['PORT'] = '7860'
print(f"Runtime: {'GPU — ' + torch.cuda.get_device_name(0) if has_gpu else 'CPU'}")
print('Whisper small and NLLB-200 distilled 600M weights download on first use and are cached for this runtime.')

In [ ]:
from app import build_app

demo = build_app().queue(default_concurrency_limit=1, max_size=8)
demo.launch(server_name='0.0.0.0', server_port=7860, share=True, show_error=True)

## Notes

- Copy the `https://....gradio.live` URL printed above and paste it into the browser studio's **Gradio backend URL**, then click **Test connection**.
- Model weights are cached in the runtime after their first download. A T4 GPU uses fp16 for NLLB translation and faster-whisper; CPU fallback can be slow.
- The browser extracts audio and keeps the original media local; only the WAV is sent to this backend. MP4 muxing happens in the browser with the original video stream copied without re-encoding.
- Edge TTS needs internet access and sends translated text to Microsoft's service.
- Download results before the Colab runtime ends. Stop the runtime when finished.